# Práctica 5 — Análisis de datos en tiempo real

**Estudiante:** Brailin Leonardo Méndez Guillermo  
**Objetivo:** Simular un flujo de transacciones bancarias, calcular indicadores móviles y detectar anomalías en tiempo real.


In [ ]:
import random, time
from collections import deque
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt

random.seed(2026)
ventana = deque(maxlen=25)
eventos = []


In [ ]:
def generar_transaccion(i):
    monto = round(random.lognormvariate(5.3, 0.8), 2)
    if i in (18, 42):
        monto *= 12  # anomalía controlada para demostrar la detección
    return {"instante": datetime.now(), "monto": round(monto,2),
            "canal": random.choice(["web","móvil","ATM"])}

for i in range(60):
    tx = generar_transaccion(i)
    ventana.append(tx["monto"])
    media = sum(ventana)/len(ventana)
    desviacion = pd.Series(ventana).std(ddof=0) or 1
    tx["media_movil"] = round(media,2)
    tx["anomalia"] = tx["monto"] > media + 3*desviacion and len(ventana) >= 10
    eventos.append(tx)
    if tx["anomalia"]:
        print(f"ALERTA {tx['instante']:%H:%M:%S}: RD${tx['monto']:,.2f}")
    time.sleep(0.02)

df = pd.DataFrame(eventos)
df.tail()


In [ ]:
resumen = df.groupby('canal').agg(
    transacciones=('monto','size'),
    monto_promedio=('monto','mean'),
    monto_total=('monto','sum'),
    anomalias=('anomalia','sum')
).round(2)
resumen


In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df.index, df['monto'], label='Monto')
plt.plot(df.index, df['media_movil'], label='Media móvil', linewidth=2)
anom = df[df['anomalia']]
plt.scatter(anom.index, anom['monto'], color='red', label='Anomalía', zorder=3)
plt.title('Monitoreo simulado de transacciones en tiempo real')
plt.xlabel('Evento'); plt.ylabel('Monto RD$'); plt.legend(); plt.grid(alpha=.25)
plt.show()


### Conclusión
El procesamiento por ventanas permite actualizar indicadores sin esperar el cierre de un lote. La regla de tres desviaciones identifica operaciones excepcionalmente altas que deberían revisarse, aunque no prueba por sí sola la existencia de fraude.